# Thực nghiệm 2: PARAFAC2 trên dữ liệu EEG

PARAFAC2 là một dạng tổng quát của CP decomposition, cho phép một chiều của tensor có độ dài khác nhau giữa các ma trận (slices). Đặc tính này rất phù hợp với dữ liệu EEG khi mỗi subject có số lượng epochs/trials hợp lệ khác nhau sau quá trình loại bỏ nhiễu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_eeg_raw, build_parafac2_slices, build_tensor
from src.parafac2 import parafac2_als
from src.cp_als import cp_als
from src.visualize import plot_parafac2_results, plot_parafac2_vs_cp_spatial

import mne
mne.set_log_level('WARNING')

## 1. Chuẩn bị dữ liệu cho PARAFAC2

Khác với `02_eeg_cp.ipynb` nơi chúng ta tính trung bình các trial, ở đây chúng ta giữ lại thông tin của từng trial. Mỗi subject sẽ tạo thành một slice kích thước `(n_trials, n_channels)`. Số lượng `n_trials` sẽ khác nhau tùy subject.

In [ ]:
n_subjects = 5  # Dùng 5 subjects để demo nhanh PARAFAC2
subjects = list(range(1, n_subjects + 1))
runs = [4, 8, 12]

raws = load_eeg_raw(subjects, runs, verbose=False)

# Xây dựng các slice cho PARAFAC2
slices, subj_ids, info = build_parafac2_slices(
    raws, 
    l_freq=8.0, h_freq=30.0,
    tmin=-0.5, tmax=2.5,
    verbose=True
)

## 2. Chạy PARAFAC2-ALS

Chúng ta sẽ phân rã bộ slices với rank $R = 4$.

In [ ]:
R = 4
print(f"Chạy PARAFAC2 với rank = {R}...")
V, weights, projections, fit_info = parafac2_als(slices, rank=R, max_iter=100, tol=1e-5, verbose=True)

## 3. Trực quan hóa Kết quả

Trong kết quả của PARAFAC2:
- **V** (Shared Spatial Matrix): là không gian các kênh (channels) chung cho toàn bộ dataset.
- **Weights**: Trọng số thể hiện mức độ biểu hiện của component ở mỗi subject.
- **Projections**: Chứa thông tin về trial, đặc trưng riêng cho từng subject.

In [ ]:
plot_parafac2_results(V, weights, projections, channel_names=info['channel_names'], subject_ids=subj_ids)
plt.show()

## 4. So sánh Spatial Component: CP vs PARAFAC2

Chạy CP-ALS trên dữ liệu đã trung bình hóa (như ở notebook 02) và so sánh với PARAFAC2.

In [ ]:
tensor, _, _ = build_tensor(raws, l_freq=8.0, h_freq=30.0, tmin=-0.5, tmax=2.5, average_trials=True, verbose=False)
_, f0, f1, f2 = cp_als(tensor, rank=R, max_iter=100, tol=1e-5, verbose=False)
cp_factors = [f0, f1, f2]

plot_parafac2_vs_cp_spatial(cp_factors, V, channel_names=info['channel_names'])
plt.show()